## Q1: Build the Personalized Knowledge Base

In [1]:

import pandas as pd
import re

roll_number = input("Enter your college roll number: ").strip()

if len(roll_number) < 2 or not roll_number[-2:].isdigit():
    raise ValueError("Please enter a roll number ending with at least two digits.")

last_two_digits = roll_number[-2:]
categories = ["billing", "account", "general"]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

generated_entries = []

for digit in last_two_digits:
    digit_value = int(digit)
    category = categories[digit_value % 3]

    if category == "billing":
        question = "how do I update my registered mobile number"
        answer = "You can update your registered mobile number from your account settings."
        keywords = "mobile number update phone registered"
    elif category == "account":
        question = "how do I change my account email"
        answer = "You can change your account email from the profile settings."
        keywords = "email change profile account"
    else:
        question = "where can I contact customer support"
        answer = "You can contact customer support during working hours."
        keywords = "contact support help service"

    generated_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

faq_data = pd.DataFrame(fixed_entries + generated_entries)

print("Last two digits:", last_two_digits)
print("Final 6-row DataFrame:")
display(faq_data)


Enter your college roll number: 1024170041
Last two digits: 41
Final 6-row DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do I change my account email,You can change your account email from the pro...,email change profile account,account
5,how do I change my account email,You can change your account email from the pro...,email change profile account,account


## Q2: Generate and Score a Hypothesis

In [2]:

def clean_text(text):
    return set(re.findall(r"[a-zA-Z0-9]+", text.lower()))


def score_query(query, entry):
    query_words = clean_text(query)
    entry_words = clean_text(
        entry["question"] + " " + entry["keywords"] + " " + entry["category"]
    )

    matched_words = query_words.intersection(entry_words)
    confidence = len(matched_words) / max(len(query_words), 1)

    return confidence


def generate_hypothesis(query, df):
    results = []

    for index, entry in df.iterrows():
        confidence = score_query(query, entry)

        if confidence > 0:
            results.append({
                "index": index,
                "question": entry["question"],
                "answer": entry["answer"],
                "category": entry["category"],
                "confidence": round(confidence, 3)
            })

    results.sort(key=lambda item: item["confidence"], reverse=True)

    return pd.DataFrame(results)


query = input("Enter your query: ")
hypothesis_results = generate_hypothesis(query, faq_data)

if hypothesis_results.empty:
    print("No matching entries found.")
else:
    display(hypothesis_results)


Enter your query: what is the annual fee


,index,question,answer,category,confidence
0,0,what is the annual fee,The annual fee is Rs 500.,billing,1.0
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.4
2,2,what are your working hours,We are open 9 AM to 5 PM.,general,0.2


## Q3: Find Questions from the Same Category

In [3]:

def same_category(category_name, df):
    matching_entries = df[df["category"].str.lower() == category_name.lower()]
    return matching_entries[["question", "answer", "keywords", "category"]]


category_name = input("Enter a category name: ")
category_results = same_category(category_name, faq_data)

if category_results.empty:
    print("No entries found for this category.")
else:
    display(category_results)


Enter a category name: general


,question,answer,keywords,category
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general


## Q4: Add a New Keyword and Save the DataFrame

In [4]:


print("Choose an entry to update:")
display(faq_data[["question", "category"]])

entry_number = int(input("Enter the DataFrame index of the entry: "))

if entry_number not in faq_data.index:
    raise ValueError("Invalid DataFrame index.")

new_keyword = input("Enter a new keyword: ").strip()

if new_keyword:
    old_keywords = faq_data.loc[entry_number, "keywords"]
    faq_data.loc[entry_number, "keywords"] = old_keywords + " " + new_keyword

csv_file_name = f"{roll_number}_faq_data.csv"
faq_data.to_csv(csv_file_name, index=False)

print("Updated DataFrame:")
display(faq_data)
print("File saved as:", csv_file_name)


Choose an entry to update:


,question,category
0,what is the annual fee,billing
1,how to reset password,account
2,what are your working hours,general
3,how can i pay the fee,billing
4,how do I change my account email,account
5,how do I change my account email,account


Enter the DataFrame index of the entry: 2
Enter a new keyword: 1
Updated DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time 1,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do I change my account email,You can change your account email from the pro...,email change profile account,account
5,how do I change my account email,You can change your account email from the pro...,email change profile account,account


File saved as: 1024170041_faq_data.csv


## Q5: Count FAQ Entries in Each Category

In [5]:

category_counts = faq_data.groupby("category").size().reset_index(name="number_of_entries")
display(category_counts)


,category,number_of_entries
0,account,3
1,billing,2
2,general,1


## Q6: Return All Entries Having the Highest Score

In [6]:

def generate_hypothesis_with_ties(query, df):
    results = []

    for index, entry in df.iterrows():
        confidence = score_query(query, entry)

        results.append({
            "index": index,
            "question": entry["question"],
            "answer": entry["answer"],
            "category": entry["category"],
            "confidence": round(confidence, 3)
        })

    results_df = pd.DataFrame(results)

    if results_df.empty:
        return results_df

    highest_score = results_df["confidence"].max()

    return results_df[results_df["confidence"] == highest_score].reset_index(drop=True)


tie_query = input("Enter a query to test tie handling: ")
tie_results = generate_hypothesis_with_ties(tie_query, faq_data)

if tie_results.empty:
    print("No entries available.")
else:
    print("All entries with the highest score:")
    display(tie_results)


Enter a query to test tie handling: what are your working hours
All entries with the highest score:


,index,question,answer,category,confidence
0,2,what are your working hours,We are open 9 AM to 5 PM.,general,1.0
